## Getting the Data


In [3]:
import pandas as pd
import re

In [4]:
messages = pd.read_csv(
    "data/SMSSpamCollection", delimiter="\t", names=["label", "message"]
)

In [5]:
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Data Cleaning and Preprocessing


In [6]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [7]:
porterstemmer = PorterStemmer()

In [8]:
corpus = []

for i in range(len(messages)):
    message = re.sub("[^a-zA-Z]", " ", messages["message"][i])
    message = message.lower()
    message = message.split()

    message = [
        porterstemmer.stem(word)
        for word in message
        if word not in set(stopwords.words("english"))
    ]

    message = " ".join(message)

    corpus.append(message)

## Encoding Target Variable (y)


In [9]:
import pandas as pd

# Encoding dependent variable: ham -> 0, spam -> 1
y = pd.get_dummies(messages["label"], drop_first=True)
y = y.iloc[:, 0].values

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    corpus, y, test_size=0.20, random_state=0
)

# BOW with N Grams


In [11]:
from sklearn.feature_extraction.text import CountVectorizer

In [12]:
countvectorizer = CountVectorizer(max_features=100, binary=True, ngram_range=(2, 2))

X_train = countvectorizer.fit_transform(X_train).toarray()
X_test = countvectorizer.transform(X_test).toarray()

countvectorizer.vocabulary_  # shows the words and their index in the array

{'great day': np.int64(43),
 'free call': np.int64(32),
 'call mobil': np.int64(9),
 'last night': np.int64(57),
 'let know': np.int64(58),
 'land row': np.int64(56),
 'wat time': np.int64(97),
 'pleas call': np.int64(70),
 'call custom': np.int64(4),
 'custom servic': np.int64(23),
 'guarante cash': np.int64(47),
 'cash prize': np.int64(14),
 'happi birthday': np.int64(51),
 'take care': np.int64(82),
 'good night': np.int64(42),
 'lt gt': np.int64(60),
 'happi new': np.int64(52),
 'new year': np.int64(65),
 'gud ni': np.int64(49),
 'nice day': np.int64(67),
 'want go': np.int64(96),
 'come back': np.int64(21),
 'dont know': np.int64(26),
 'call mobileupd': np.int64(10),
 'call optout': np.int64(11),
 'chanc win': np.int64(15),
 'miss call': np.int64(61),
 'repli call': np.int64(75),
 'tri contact': np.int64(85),
 'sorri call': np.int64(80),
 'call later': np.int64(8),
 'date servic': np.int64(24),
 'po box': np.int64(71),
 'watch tv': np.int64(98),
 'ur mob': np.int64(91),
 'txt noki

## Training Model using Naive Bayes Classifier


In [13]:
from sklearn.naive_bayes import MultinomialNB

spam_detect_model = MultinomialNB().fit(X_train, y_train)

## Model Prediction


In [14]:
y_pred = spam_detect_model.predict(X_test)

## Model Evaluation


In [15]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [16]:
confusion_m = confusion_matrix(y_test, y_pred)
confusion_m

array([[954,   1],
       [ 69,  91]])

In [17]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9372197309417041

In [18]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.93      1.00      0.96       955
        True       0.99      0.57      0.72       160

    accuracy                           0.94      1115
   macro avg       0.96      0.78      0.84      1115
weighted avg       0.94      0.94      0.93      1115

